# MiniWeb

## Objetivo

Transformar programas escritos en lenguaje **miniweb** (español simple) en código HTML.

## Alcance

- Reconocer seis sentencias del lenguaje mweb: `titulo`, `parrafo`, `enlace`, `imagen`, `tabla` y `fila`.
- Permitir, al final de cualquier sentencia, los atributos opcionales `id` y `clase`.
- Armar tablas HTML con una sentencia `tabla` (encabezados) seguida de una o más sentencias `fila` (datos).
- Generar un documento HTML completo respetando el orden del programa fuente.
- Usar un LLM local (Ollama, modelo `llama3.2`) para generar las etiquetas `h1`, `p`, `a` e `img`.
- Verificar la respuesta del LLM y usar un respaldo si la respuesta no es válida o el LLM no está disponible.
- Permitir que los textos contengan símbolos como `<` o `&` sin romper el HTML generado.
- Detectar y reportar, con número de línea, errores **léxicos**, **sintácticos** y **semánticos**.
- No generar HTML si el programa tiene errores.
- Fuera de alcance: reglas de estilo CSS (solo se asignan `id` y `class`), listas y anidamiento de elementos.

## Ejemplo

**Programa fuente:**

```
titulo "Mi página" id "principal"
parrafo "Hola mundo" clase "destacado"
enlace "Google" "https://google.com" clase "link externo"
imagen "logo.png" "Logo de la página" id "logo"
tabla "Nombre" "Edad" id "alumnos" clase "tabla-azul"
fila "Ana" "25"
fila "Luis" "30"
parrafo "Fin de la página"
```

**Programa destino:**

```html
<html>
<head><meta charset="UTF-8"></head>
<body>
<h1 id="principal">Mi página</h1>
<p class="destacado">Hola mundo</p>
<a href="https://google.com" class="link externo">Google</a>
<img src="logo.png" alt="Logo de la página" id="logo">
<table id="alumnos" class="tabla-azul">
<tr><th>Nombre</th><th>Edad</th></tr>
<tr><td>Ana</td><td>25</td></tr>
<tr><td>Luis</td><td>30</td></tr>
</table>
<p>Fin de la página</p>
</body>
</html>
```

## Especificaciones léxicas

| Lexema | Token | Patrón |
|---|---|---|
| `titulo` | TITULO | `titulo` |
| `parrafo` | PARRAFO | `parrafo` |
| `enlace` | ENLACE | `enlace` |
| `imagen` | IMAGEN | `imagen` |
| `tabla` | TABLA | `tabla` |
| `fila` | FILA | `fila` |
| `id` | ID | `id` |
| `clase` | CLASE | `clase` |
| `"texto"` | TEXTO | `"[^"\n]*"` |

Las palabras reservadas se escriben en minúscula y sin tildes. Se ignoran espacios, tabulaciones y saltos de línea (estos últimos solo se usan para contar líneas).

**Errores léxicos que se detectan:** cualquier palabra o símbolo que no coincide con ningún patrón. Por ejemplo, un carácter ilegal (`@`), una palabra que no es reservada (`subtitulo`) o una comilla sin cerrar (`"Hola`).

## Especificaciones sintácticas

```
<Programa>  ::= <Sentencia>
<Programa>  ::= <Programa> <Sentencia>

<Sentencia> ::= TITULO TEXTO <Atributos>
<Sentencia> ::= PARRAFO TEXTO <Atributos>
<Sentencia> ::= ENLACE TEXTO TEXTO <Atributos>
<Sentencia> ::= IMAGEN TEXTO TEXTO <Atributos>
<Sentencia> ::= TABLA <Textos> <Atributos>
<Sentencia> ::= FILA <Textos> <Atributos>

<Textos>    ::= TEXTO
<Textos>    ::= <Textos> TEXTO

<Atributos> ::= λ
<Atributos> ::= <Atributos> <Atributo>
<Atributo>  ::= ID TEXTO
<Atributo>  ::= CLASE TEXTO
```

- Se usa recursión a izquierda en `<Programa>`, `<Textos>` y `<Atributos>` para que los elementos se reduzcan en el mismo orden en que aparecen en el programa fuente.
- `λ` significa que los atributos son **opcionales**.
- `<Textos>` es una lista de uno o más textos: sirve para tablas con cualquier cantidad de columnas.

## Especificaciones semánticas

| Regla | Acciones semánticas |
|---|---|
| `<Programa> ::= <Sentencia>` | Una sentencia |
| `<Programa> ::= <Programa> <Sentencia>` | Muchas sentencias, en el orden del programa fuente |
| `<Sentencia> ::= TITULO TEXTO <Atributos>` | Valida que TEXTO no sea vacío. Trad: `<h1 atributos>TEXTO</h1>` |
| `<Sentencia> ::= PARRAFO TEXTO <Atributos>` | Valida que TEXTO no sea vacío. Trad: `<p atributos>TEXTO</p>` |
| `<Sentencia> ::= ENLACE TEXTO1 TEXTO2 <Atributos>` | Valida que TEXTO1 no sea vacío y que TEXTO2 (URL) comience con `http://` o `https://`. Trad: `<a href="TEXTO2" atributos>TEXTO1</a>` |
| `<Sentencia> ::= IMAGEN TEXTO1 TEXTO2 <Atributos>` | Valida que TEXTO1 (ruta) termine en `.png`, `.jpg`, `.jpeg`, `.gif`, `.webp` o `.svg` y que TEXTO2 no sea vacío. Trad: `<img src="TEXTO1" alt="TEXTO2" atributos>` |
| `<Sentencia> ::= TABLA <Textos> <Atributos>` | Valida que ningún encabezado sea vacío y recuerda la cantidad de columnas. Trad: `<table atributos>` + `<tr><th>T1</th><th>T2</th>...</tr>` |
| `<Sentencia> ::= FILA <Textos> <Atributos>` | Valida que haya una tabla abierta y que la fila tenga tantas celdas como columnas. Trad: `<tr atributos><td>T1</td><td>T2</td>...</tr>` |
| `<Textos> ::= TEXTO` | Lista con un texto |
| `<Textos> ::= <Textos> TEXTO` | Agrega el texto a la lista |
| `<Atributos> ::= λ` | Lista vacía |
| `<Atributos> ::= <Atributos> <Atributo>` | Agrega el atributo a la lista |
| `<Atributo> ::= ID TEXTO` | Trad: `id="TEXTO"` |
| `<Atributo> ::= CLASE TEXTO` | Trad: `class="TEXTO"` |

**Reglas semánticas globales:**
1. La primera sentencia del programa debe ser un `titulo`.
2. Una `fila` solo puede ir inmediatamente después de una `tabla` o de otra `fila`. Cualquier otra sentencia **cierra** la tabla (`</table>`).
3. Cada `fila` debe tener tantas celdas como columnas tiene la `tabla`.

**Reglas de los atributos** (se validan al reconocer la sentencia):
- Un `id` y una `clase` deben empezar con una letra y usar solo letras, números, `-` o `_`. La `clase` puede tener varios nombres separados por espacios.
- No puede haber dos sentencias con el mismo `id`.
- No se puede repetir `id` ni `clase` en la misma sentencia.

**Errores semánticos que se detectan:** texto vacío, URL inválida, extensión de imagen inválida, programa que no comienza con `titulo`, `fila` fuera de una tabla, fila con distinta cantidad de celdas que la tabla, `id` o `clase` inválidos, `id` duplicado y atributo repetido.

**Decisiones de diseño:**
- Las acciones semánticas validan y, en la misma regla, agregan la etiqueta al HTML. Ollama se consulta solo mientras no haya errores, para no gastar llamadas en programas incorrectos.
- Las tablas **no usan anidamiento**: `tabla` y `fila` son sentencias comunes y el parser recuerda si hay una tabla abierta.
- Ollama genera las etiquetas de contenido (`h1`, `p`, `a`, `img`). La estructura de la tabla (`table`, `tr`, `th`, `td`) la arma el programa directamente.

## Implementación

In [4]:
%%file traductor_mw/lexer_mw.py
import re
import ply.lex as lex

# Lista de tokens del lenguaje (coincide con la grilla de especificaciones léxicas)
tokens = (
    'TITULO', 'PARRAFO', 'ENLACE', 'IMAGEN',
    'TABLA', 'FILA', 'ID', 'CLASE', 'TEXTO'
)

# Acá se guardan TODOS los errores del programa (léxicos, sintácticos y semánticos)
errores = []

# Palabras reservadas: cada una es un token con su propio patrón.
# _t define el token
# \b significa "fin de palabra": así 'tituloX' no se reconoce como 'titulo'.
t_TITULO  = r'titulo\b'
t_PARRAFO = r'parrafo\b'
t_ENLACE  = r'enlace\b'
t_IMAGEN  = r'imagen\b'
t_TABLA   = r'tabla\b'
t_FILA    = r'fila\b'
t_ID      = r'id\b'
t_CLASE   = r'clase\b'

# Texto entre comillas dobles. se le sacan las comillas: "Hola" -> Hola
def t_TEXTO(t):
    r'"[^"\n]*"' # patron
    t.value = t.value[1:-1] # recorta 1er y ultimo caracter del texto
    return t # entrega el token al parser

t_ignore = ' \t\r' # espacios y tabulaciones se ignoran

def t_newline(t):
    r'\n+'
    t.lexer.lineno += len(t.value)   # solo cuenta las líneas (para informar dónde está cada error)

# ply llama esta funcion cuando hay un error
def t_error(t):
    palabra = re.split(r'\s', t.value, maxsplit=1)[0] or t.value[0] # corta ese texto en el primer espacio y se queda con la primera parte, la “palabra” que contiene el error. or t.value[0] es un respaldo: si esa primera parte quedara vacía, toma el primer carácter.
    errores.append(f"Error léxico (línea {t.lexer.lineno}): '{palabra}' no es válido")
    t.lexer.skip(len(palabra))       # descarta esa cantidad de caracteres y se sigue leyendo

# construir scanner
lexer = lex.lex()


Writing traductor_mw/lexer_mw.py


### Generador de etiquetas con LLM (`ag_generar_tag_html.py`)

Para cada sentencia se le pide a Ollama que devuelva **solo** la etiqueta HTML, con sus atributos. Como un LLM no es 100 % determinista (por ejemplo, puede agregar un punto al final del texto), la respuesta solo se acepta si empieza y termina con la etiqueta pedida, el texto visible es **exactamente** el del programa y tiene todos los atributos. Si no cumple, o si Ollama no está disponible, se usa un respaldo y se muestra un aviso.

In [ ]:
%%file traductor_mw/ag_generar_tag_html.py
import re
import ollama
from html import escape

# Convierte una lista de atributos en el texto que va dentro de la etiqueta. ['id="x"', 'class="y"'] -> ' id="x" class="y"'
def attrs_html(atributos):
    return ''.join(' ' + a for a in atributos)

# Le pide a Ollama el tag HTML. Si no responde o responde mal, usa un respaldo.
def generar_tag_html(tag, texto, atributos):
    texto = escape(texto) # convierte < > & " a &lt; &gt; &amp; &quot; para no romper el codigo
    attrs = attrs_html(atributos) # lista de atributos ya convertida en texto.
    cierre = '' if tag == 'img' else f'</{tag}>' # etiqueta de cierre. img no se cierra
    respaldo = f'<{tag}{attrs}>{texto}{cierre}' # el HTML correcto que esperamos recibir

    pedido = f'Genera el tag {tag}' # arma la frasse que se le pide al modelo
    if attrs:
        pedido += f' con los atributos{attrs}'
    if texto:
        pedido += f' para el texto: {texto}'

    # aca se llama a ollama: le manda un mensaje pidiendo la etiqueta HTML y guarda lo que le responde
    try:
        respuesta = ollama.chat(
            model='llama3.2',
            messages=[ # funciona como un chat
                { # emssages es una lista con un msj, con dos datos: role quien habla (user, mi programa) y content (el texto, las instrucciones estrictas mas el pedido)
                    'role': 'user',
                    'content': f'''
                    Devuelve únicamente el HTML.
                    Sin explicaciones.
                    Sin markdown.
                    Sin texto adicional.
                    {pedido}
                    '''
                }
            ],
            options={'temperature': 0} # pide al modelo que sea lo mas predecible posible
        )
        tag_ollama = respuesta['message']['content'].strip() # La respuesta viene anidada: respuesta tiene una clave message, que tiene una clave content con el texto. strip() saca espacios y saltos de los costados.

        # Se acepta la respuesta de Ollama solo si:
        if (tag_ollama.startswith(f'<{tag}')                       # empieza con el tag pedido
                and tag_ollama.endswith(cierre or '>')             # y lo cierra
                and re.sub(r'<[^>]*>', '', tag_ollama) == texto    # el texto visible es EXACTAMENTE el del programa
                and all(a in tag_ollama for a in atributos)):      # y tiene todos los atributos
            return tag_ollama
    except Exception:
        pass                                           # Ollama no disponible

    print(f'[aviso] se usa el respaldo: {respaldo}')
    return respaldo


Writing traductor_mw/ag_generar_tag_html.py


### Parser con acciones semánticas (`parser_mw.py`)

In [ ]:
%%file traductor_mw/parser_mw.py
import re
import ply.yacc as yacc
from html import escape
from lexer_mw import tokens, errores
from ag_generar_tag_html import generar_tag_html, attrs_html

# ---------- datos que se van armando mientras el parser lee el programa ----------
html = ['<html>', '<head><meta charset="UTF-8"></head>', '<body>']   # variable lista donde se va construyendo el HTML resultante
ids = set() # guarda ids ya usados para detectar repetidos
estado = {'sentencias': 0, 'columnas': 0} # diccionario con cuantas sentencias fueron procesadas y recuerda cuajntas columnas tiene la tabla que esta abierta 

# Formato válido de id y clases: almacena ER para validar atributos. empieza con letra y sigue con letras, números, - o _ (la clase puede tener varios nombres)
FORMATO = {'id':    r'^[A-Za-z][\w-]*$',
           'class': r'^[A-Za-z][\w-]*( [A-Za-z][\w-]*)*$'}

# funciones auxiliares de ayuda para las acciones semánticas, funciones que no son reglas de la gramática

def error_semantico(linea, mensaje): # Guarda un mensaje de error semántico con su línea
    errores.append(f'Error semántico (línea {linea}): {mensaje}') # se agrega a errores

def texto_no_vacio(texto, campo, linea): # Si el texto está vacío anota el error
    if texto.strip() == '':
        error_semantico(linea, f'{campo} vacío')

# Valida los id y class de una sentencia y los devuelve listos para el HTML: ['id="x"', 'class="y"']
def validar_atributos(lista):
    atributos, vistos = [], []
    for nombre, valor, linea in lista: # Recibe los atributos de una sentencia: una lista de tuplas (nombre, valor, línea). Prepara dos listas vacías y recorre cada tupla, desarmándola en tres variables.
        if nombre in vistos: # ¿ya se usó este atributo en esta sentencia?, x ejemplo si pones dos id
            error_semantico(linea, f"'{nombre}' repetido en la misma sentencia")
        elif not re.match(FORMATO[nombre], valor): # ¿tiene el formato correcto de id o class?
            error_semantico(linea, f"{nombre} inválido '{valor}'")
        elif nombre == 'id' and valor in ids: # ¿el id ya lo usó otra sentencia?
            error_semantico(linea, f"id duplicado '{valor}'")
        vistos.append(nombre) # se anota el atributo como visto
        if nombre == 'id':
            ids.add(valor) # se guarda el id en el conjunto de ids definido al principio
        atributos.append(f'{nombre}="{valor}"') # se agrega a la lista el atributo ya listo para el HTML
    return atributos

# comprobaciones comunes a todas las sentencias.
# recibe tipo de setencia, nro de linea y atributos, devuelve los atributos listos para el HTML.
# basicamente comprueba si el título cumple las condiciones generales, controla el estado de las tablas y valida sus atributos
def inicio(tipo, linea, atributos):
    if estado['sentencias'] == 0 and tipo != 'titulo': # Si es la primera y no es un titulo, error
        error_semantico(linea, 'el programa debe comenzar con un titulo')
    estado['sentencias'] += 1 # suma una sentencia al contador
    if tipo != 'fila' and estado['columnas'] > 0: # comprueba el cierre de tablas
        html.append('</table>')
        estado['columnas'] = 0
    return validar_atributos(atributos) # Valida los atributos y los devuelve listos para el HTML

def agregar(tag, texto, atributos): # agrega una etiqueta HTML generada por Ollama, solo si todavía no hubo errores
    if not errores:
        html.append(generar_tag_html(tag, texto, atributos)) # llama a la función ag_generar_tag_html.py y agrega el resultado a la lista html

def fila_html(celda, textos, atributos): # Arma una fila de tabla html (sin Ollama) por cada texto crea una celda y las une.
    celdas = ''.join(f'<{celda}>{escape(t)}</{celda}>' for t in textos)
    return f'<tr{attrs_html(atributos)}>{celdas}</tr>'

# ---------- gramática (GIC) y acciones semánticas ----------
# En cada regla: p[1], p[2]... son las partes de la regla (de izquierda a derecha)
#                p.lineno(1) es la línea de la primera parte

def p_programa(p): # regla para conocer un programa completo
    '''
    programa : programa sentencia
             | sentencia
    '''

def p_titulo(p): # regla para reconocer un titulo
    'sentencia : TITULO TEXTO atributos'
    linea = p.lineno(1) # obtiene nro de linea, 1= 1er elemento (titulo)
    attrs = inicio('titulo', linea, p[3]) # llama a al funcion inicio, le pasa tipo de sent (titulo), nro linea y atributos q estan en p3
    texto_no_vacio(p[2], 'texto del titulo', linea) # comprueba que el texto no este vacio
    agregar('h1', p[2], attrs) # Solicita la generación de la etiqueta HTML correspondiente
    # esto es lo que generaría por ej: <h1 id="principal">Mi página</h1>

def p_parrafo(p):
    'sentencia : PARRAFO TEXTO atributos'
    linea = p.lineno(1)
    attrs = inicio('parrafo', linea, p[3]) # pasa parrafo, nro de linea y los atributos p3
    texto_no_vacio(p[2], 'texto del parrafo', linea)
    agregar('p', p[2], attrs)

def p_enlace(p):
    'sentencia : ENLACE TEXTO TEXTO atributos'
    linea = p.lineno(1)
    attrs = inicio('enlace', linea, p[4])
    texto_no_vacio(p[2], 'texto del enlace', linea)
    if not re.match(r'^https?://\S+$', p[3]): # valida la URL. debe empezar con http:// o https://
        error_semantico(linea, f"URL inválida '{p[3]}' (debe empezar con http:// o https://)")
    agregar('a', p[2], [f'href="{escape(p[3])}"'] + attrs) # genera el html. el href contiene la url
    # por ej generaría <a href="https://google.com" class="link externo">Google</a>

def p_imagen(p):
    'sentencia : IMAGEN TEXTO TEXTO atributos'
    linea = p.lineno(1)
    attrs = inicio('imagen', linea, p[4]) # comprueba en gral y valida atributos
    if not re.search(r'\.(png|jpe?g|gif|webp|svg)$', p[2], re.IGNORECASE): # la ruta debe terminar en una extensión de imagen permitida. igorecase es ppara aceptar mayuscula y minuscula
        error_semantico(linea, f"extensión de imagen inválida '{p[2]}'")
    texto_no_vaci
    o(p[3], 'descripción de la imagen', linea) # comprueba que la descripción no esté vacía
    agregar('img', '', [f'src="{escape(p[2])}"', f'alt="{escape(p[3])}"'] + attrs) # generación de la etiqueta HTML

def p_tabla(p):
    'sentencia : TABLA textos atributos'
    linea = p.lineno(1)
    attrs = inicio('tabla', linea, p[3]) # comprueba y valida atributos de la tabla
    for encabezado in p[2]: # recorre todos los encabezados de la tabla
        texto_no_vacio(encabezado, 'encabezado de columna', linea) # comprueba que ninguno esté vacío
    estado['columnas'] = len(p[2])                     # se recuerda cuántas columnas tiene la tabla. necesita recordar cuántas columnas tiene la tabla para validar las filas que aparezcan después.
    html.append(f'<table{attrs_html(attrs)}>')         # los atributos de la tabla van en <table>
    html.append(fila_html('th', p[2], []))             # agrega la apertura de la tabla q contiene los encabezados

def p_fila(p):
    'sentencia : FILA textos atributos'
    linea = p.lineno(1)
    attrs = inicio('fila', linea, p[3]) # valida atributos de fila
    if estado['columnas'] == 0:                        # si columnas vale 0 no hay una tabla abierta
        error_semantico(linea, "'fila' debe ir después de una 'tabla' o de otra 'fila'")
    elif len(p[2]) != estado['columnas']: # si hay tabla, Comprueba que la cantidad de celdas de la fila coincida con la cantidad de columnas de la tabla.
        error_semantico(linea, f'la fila tiene {len(p[2])} celdas y la tabla tiene {estado["columnas"]} columnas')
    html.append(fila_html('td', p[2], attrs)) # construye la fila con sus celdas

# FUNCIONES QUE CONSTRUYEN LISTAS

def p_textos_uno(p): # indica que una lista de textos puede estar formada por un único texto, x ejemplo "nombre"
    'textos : TEXTO'
    p[0] = [p[1]]

def p_textos_varios(p): # esta regla dice qe una lista de textos puede estar formada por otra lista de textos más un texto nuevo
    'textos : textos TEXTO'
    p[0] = p[1] + [p[2]] # por ej si p1 es nombre y p2 edad, p0= nombre + edad

def p_atributos_vacio(p): # representa los atributos opcionales id/class, por ej titulo "Mi página"
    'atributos :'
    p[0] = []

def p_atributos_varios(p): # permite agregar atributos a una lista existente
    'atributos : atributos atributo'
    p[0] = p[1] + [p[2]]

def p_atributo_id(p): # reconoce un atributo id
    'atributo : ID TEXTO'
    p[0] = ('id', p[2], p.lineno(1))

def p_atributo_clase(p): # Reconoce el atributo clase y lo transforma en class
    'atributo : CLASE TEXTO'
    p[0] = ('class', p[2], p.lineno(1))

def p_error(p): # se ejecuta cuando el parser encuentra un error sintáctico.
    if p is None:
        raise SyntaxError('Error sintáctico: fin del programa inesperado')
    raise SyntaxError(f"Error sintáctico (línea {p.lineno}): token inesperado {p.type} ('{p.value}')")

# construir parser
parser = yacc.yacc(debug=False, write_tables=False)


Writing traductor_mw/parser_mw.py


### Programa principal (`main.py`)

In [7]:
%%file traductor_mw/main.py
import sys
from lexer_mw import errores
from parser_mw import parser, html, estado

entrada = open(sys.argv[1], 'r', encoding='utf-8').read()

try:
    parser.parse(entrada)       # hace todo el análisis: el parser le va pidiendo tokens al scanner, aplica las reglas y ejecuta las acciones semánticas (acá se arma el HTML)
except SyntaxError as e:        # error sintáctico
    errores.append(str(e))

if errores: # es verdadero si la lista no está vacía. si hay errores no se genera HTML
    for error in errores:
        print(error)
    sys.exit(1) # si hay errores, los imprime uno por línea y termina con código 1
    # por convención, 0 significa que todo bien y cualquier otro número “falló”. Los tests usan ese código

if estado['columnas'] > 0:      # si el programa terminó con una tabla abierta (la última sentencia fue una fila), se cierra
    html.append('</table>')
html.append('</body>')
html.append('</html>')

archivo = open(f'{sys.argv[1]}.html', 'w', encoding='utf-8') # Abre un archivo nuevo (el nombre original más .html) en modo escritura ('w')
for linea in html:
    print(linea)
    archivo.write(linea + '\n') # Por cada línea del HTML, la muestra en pantalla y la escribe con un salto de línea
archivo.close()


Writing traductor_mw/main.py


### Ejecuciones de ejemplo

**Caso feliz**

In [8]:
%%file traductor_mw/caso_feliz.mweb
titulo "Mi página" id "principal"
parrafo "Hola mundo" clase "destacado"
enlace "Google" "https://google.com" clase "link externo"
imagen "logo.png" "Logo de la página" id "logo"
tabla "Nombre" "Edad" id "alumnos" clase "tabla-azul"
fila "Ana" "25"
fila "Luis" "30"
parrafo "Fin de la página"


Writing traductor_mw/caso_feliz.mweb


In [9]:
!python traductor_mw/main.py traductor_mw/caso_feliz.mweb

[aviso] se usa el respaldo: <p>Fin de la página</p>
<html>
<head><meta charset="UTF-8"></head>
<body>
<h1 id="principal">Mi página</h1>
<p class="destacado">Hola mundo</p>
<a href="https://google.com" class="link externo">Google</a>
<img src="logo.png" alt="Logo de la página" id="logo">
<table id="alumnos" class="tabla-azul">
<tr><th>Nombre</th><th>Edad</th></tr>
<tr><td>Ana</td><td>25</td></tr>
<tr><td>Luis</td><td>30</td></tr>
</table>
<p>Fin de la página</p>
</body>
</html>


In [10]:
# Ver el HTML generado renderizado en el navegador del notebook
from IPython.display import HTML, display
display(HTML(open('traductor_mw/caso_feliz.mweb.html', encoding='utf-8').read()))

Nombre,Edad
Ana,25
Luis,30


**Caso con error sintáctico** (el `id` no tiene valor)

In [11]:
%%file traductor_mw/caso_error_sintax.mweb
titulo "Mi página" id
parrafo "Hola mundo"


Writing traductor_mw/caso_error_sintax.mweb


In [12]:
!python traductor_mw/main.py traductor_mw/caso_error_sintax.mweb

Error sintáctico (línea 2): token inesperado PARRAFO ('parrafo')


**Caso con errores semánticos** (`id` repetido y filas con distinta cantidad de celdas que la tabla)

In [13]:
%%file traductor_mw/caso_error_semantico.mweb
titulo "Mi página" id "principal"
parrafo "Hola" id "principal"
tabla "Nombre" "Edad"
fila "Ana"
fila "Luis" "30" "extra"


Writing traductor_mw/caso_error_semantico.mweb


In [14]:
!python traductor_mw/main.py traductor_mw/caso_error_semantico.mweb

Error semántico (línea 2): id duplicado 'principal'
Error semántico (línea 4): la fila tiene 1 celdas y la tabla tiene 2 columnas
Error semántico (línea 5): la fila tiene 3 celdas y la tabla tiene 2 columnas


**Caso con errores léxicos** (carácter ilegal, palabra no reconocida y comilla sin cerrar)

In [15]:
%%file traductor_mw/caso_error_lexico.mweb
titulo "Mi página" @
subtitulo "Hola"
parrafo "Sin cerrar

Writing traductor_mw/caso_error_lexico.mweb


In [16]:
!python traductor_mw/main.py traductor_mw/caso_error_lexico.mweb

Error léxico (línea 1): '@' no es válido
Error léxico (línea 2): 'subtitulo' no es válido
Error sintáctico (línea 2): token inesperado TEXTO ('Hola')


## TestSuite

Se definen **12 casos de prueba automatizados** con `pytest`. Cada uno escribe un programa mweb en un archivo, ejecuta `main.py` y verifica el resultado:

| Categoría | Cantidad |
|---|---|
| Programas válidos | 4 |
| Errores léxicos | 2 |
| Errores sintácticos | 2 |
| Errores semánticos | 4 |

Los programas válidos usan el LLM real (Ollama). Las aserciones verifican los elementos esenciales de cada etiqueta y no el texto exacto, porque el modelo puede variar detalles de formato.

In [ ]:
%%file traductor_mw/tests.py
import os
import subprocess
import sys
from pathlib import Path

MAIN = str(Path(__file__).parent / 'main.py')


def ejecutar(programa, tmp_path):
    # Escribe el programa en un archivo y ejecuta main.py.
    # Devuelve (código de salida, texto): el HTML generado, o los errores si no se generó.
    archivo = tmp_path / 'prueba.mweb'
    archivo.write_text(programa, encoding='utf-8')
    r = subprocess.run([sys.executable, MAIN, str(archivo)], capture_output=True, text=True,
                       encoding='utf-8', env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    generado = Path(str(archivo) + '.html')
    return r.returncode, generado.read_text(encoding='utf-8') if generado.exists() else r.stdout


# ================= Programas válidos =================

def test_valido_pagina_completa(tmp_path):
    codigo, html = ejecutar('titulo "Mi página"\nparrafo "Hola mundo"\n'
                            'enlace "Google" "https://google.com"\nimagen "logo.png" "Logo"', tmp_path)
    assert codigo == 0
    assert html.index('<h1') < html.index('<p') < html.index('<a ') < html.index('<img')   # mismo orden del programa
    assert 'href="https://google.com"' in html
    assert 'src="logo.png"' in html

def test_valido_id_y_clase(tmp_path):
    codigo, html = ejecutar('titulo "Hola" id "principal" clase "grande destacado"', tmp_path)
    assert codigo == 0
    assert 'id="principal"' in html
    assert 'class="grande destacado"' in html

def test_valido_tabla(tmp_path): # TEST COMPLEJO
    codigo, html = ejecutar('titulo "Alumnos"\ntabla "Nombre" "Edad" id "t1" clase "azul"\n'
                            'fila "Pepita" "25"\nfila "Luis" "30"\nparrafo "Fin"', tmp_path)
    assert codigo == 0 # chequea que el programa terminó bien
    assert '<table id="t1" class="azul">' in html # Los atributos de la tabla (id y clase) están en la etiqueta <table>
    assert html.count('<tr') == 3 and html.count('<th') == 2 and html.count('<td') == 4 # La estructura: 3 filas (1 de encabezado y 2 de datos), 2 celdas de encabezado (<th) y 4 de datos (<td)
    assert html.index('</table>') < html.index('<p') # la tabla se cierra antes del párrafo

def test_valido_escapa_caracteres_especiales(tmp_path):
    codigo, html = ejecutar('titulo "a < b & c"', tmp_path)
    assert codigo == 0
    assert 'a &lt; b &amp; c' in html


# ================= Errores léxicos =================

def test_lexico_caracter_ilegal(tmp_path): # TEST SENCILLO
    codigo, salida = ejecutar('titulo "Hola" @', tmp_path)
    assert codigo == 1
    assert 'léxico' in salida and "'@'" in salida # La salida sería: Error léxico (línea 1): '@' no es válido.

def test_lexico_comilla_sin_cerrar(tmp_path):
    codigo, salida = ejecutar('titulo "Hola"\nparrafo "Hola mundo', tmp_path)
    assert codigo == 1
    assert 'léxico' in salida and 'línea 2' in salida


# ================= Errores sintácticos =================

def test_sintactico_enlace_con_un_solo_texto(tmp_path):
    codigo, salida = ejecutar('titulo "A"\nenlace "Google"\nparrafo "x"', tmp_path)
    assert codigo == 1
    assert 'sintáctico' in salida and 'línea 3' in salida

def test_sintactico_tabla_sin_columnas(tmp_path):
    codigo, salida = ejecutar('titulo "A"\ntabla', tmp_path)
    assert codigo == 1
    assert 'sintáctico' in salida


# ================= Errores semánticos =================

def test_semantico_url_invalida(tmp_path):
    codigo, salida = ejecutar('titulo "A"\nenlace "Google" "google.com"', tmp_path)
    assert codigo == 1
    assert 'semántico' in salida and 'URL' in salida

def test_semantico_id_duplicado(tmp_path):
    codigo, salida = ejecutar('titulo "A" id "x"\nparrafo "B" id "x"', tmp_path)
    assert codigo == 1
    assert 'duplicado' in salida and 'línea 2' in salida

def test_semantico_fila_fuera_de_tabla(tmp_path):
    codigo, salida = ejecutar('titulo "A"\nfila "1" "2"', tmp_path)
    assert codigo == 1
    assert 'semántico' in salida and "'fila'" in salida

def test_semantico_fila_con_distinta_cantidad_de_celdas(tmp_path):
    codigo, salida = ejecutar('titulo "A"\ntabla "Nombre" "Edad"\nfila "Ana"', tmp_path)
    assert codigo == 1
    assert 'celdas' in salida and 'línea 3' in salida


Writing traductor_mw/tests.py


In [18]:
!pytest traductor_mw/tests.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.4, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\aixag\Downloads\ejerciciosTallerDeLenguajes\.venv-1\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\aixag\Downloads\ejerciciosTallerDeLenguajes\tp_final_traductor
plugins: anyio-4.16.0
collecting ... collected 12 items

traductor_mw/tests.py::test_valido_pagina_completa PASSED                [  8%]
traductor_mw/tests.py::test_valido_id_y_clase PASSED                     [ 16%]
traductor_mw/tests.py::test_valido_tabla PASSED                          [ 25%]
traductor_mw/tests.py::test_valido_escapa_caracteres_especiales PASSED   [ 33%]
traductor_mw/tests.py::test_lexico_caracter_ilegal PASSED                [ 41%]
traductor_mw/tests.py::test_lexico_comilla_sin_cerrar PASSED             [ 50%]
traductor_mw/tests.py::test_sintactico_enlace_con_un_solo_texto PASSED   [ 58%]
traductor_mw/tests.py::test_sintactico_tabla_sin_column

## Conclusiones

- La estructura **scanner → parser → acciones semánticas** permitió separar bien las responsabilidades: el scanner reconoce los lexemas, el parser valida el orden y las acciones semánticas controlan el significado y arman la traducción.
- PLY simplificó mucho la implementación: los tokens se describen con expresiones regulares y la gramática con reglas BNF escritas como funciones de Python.
- Los atributos opcionales `id` y `clase` se resolvieron con una regla vacía (`λ`) y una lista recursiva, y las tablas **sin anidamiento**: `tabla` y `fila` son sentencias comunes y el parser recuerda si hay una tabla abierta.
- Los errores léxicos, sintácticos y semánticos son de naturaleza distinta (un símbolo inválido, un orden inválido y un valor inválido), y la TestSuite permitió comprobar que cada tipo se detecta y se informa con su número de línea.
- Usar un LLM dentro de la traducción funciona, pero **no es determinista**: por eso se agregaron un prompt estricto, temperatura 0, una verificación de la respuesta y un respaldo. Sin esas medidas los tests podrían fallar de manera aleatoria.
- Posibles mejoras: listas, anidamiento de elementos y recuperación de errores para informar varios errores sintácticos en una sola ejecución.